# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a reproducible workflow for loading and exploring the [FAIR²](https://sen.science/doi/10.71728/senscience.qs2f-h81p) dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

This dataset provides tabular clinical and molecular variables regarding second primary colorectal cancer among cancer survivors.

In [ ]:
# Ensure `mlcroissant` library is installed and up to date
!pip install --quiet mlcroissant

## 1. Data Loading
Load Croissant metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
# Display human-readable title and description
print("Title:", dataset.metadata.name)
print("Description:", dataset.metadata.description)
print("Published:", getattr(dataset.metadata, 'datePublished', 'Unknown'))

## 2. Data Overview
Review the available *record sets* and their corresponding fields (columns).

We'll display all available record set `@id`s and show fields under one as an example.

In [ ]:
# List all record set @id's from metadata
record_sets = dataset.metadata.recordSet
if not record_sets:
    raise ValueError('No record sets found in this dataset.')

print("Available Record Sets:")
for rs in record_sets:
    print(f"- {getattr(rs, '@id', '(no id)')}")

# Display fields (columns) for the first record set
first_record_set = record_sets[0]
print(f"\nFields for record set '@id': {getattr(first_record_set, '@id', None)}")
for f in getattr(first_record_set, 'field', []):
    field_id = getattr(f, '@id', None)
    name = getattr(f, 'name', '')
    dtype = getattr(f, 'dataType', '')
    print(f"- {field_id} (name: {name}, dataType: {dtype})")

## 3. Data Extraction

Extract data from each record set into a DataFrame for analysis. Use only Croissant `@id` references throughout.

In [ ]:
# Extract all record set @id's
record_set_ids = [getattr(rs, '@id', None) for rs in record_sets if getattr(rs, '@id', None) is not None]
dataframes = {}
for record_set_id in record_set_ids:
    print(f"Loading records for record_set '@id': {record_set_id}")
    recs = list(dataset.records(record_set=record_set_id))
    # Defensive: some records may be empty
    try:
        dataframes[record_set_id] = pd.DataFrame(recs)
    except Exception as e:
        print(f"Record set {record_set_id} could not be loaded as a DataFrame: {e}")

# For this dataset, the main clinical table is in the first record set:
main_record_set_id = record_set_ids[0]
print(f"\nColumns for record set '@id': {main_record_set_id}")
print(dataframes[main_record_set_id].columns.tolist())
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps: filtering records, normalizing numeric fields, and grouping/categorizing.

We'll:
- Select a numeric field (e.g., 'Age') by its `@id`
- Filter for records above a threshold
- Normalize the field
- Group by sex if present

In [ ]:
# Step 1: Identify field IDs for numeric and group fields
fields = {getattr(f, 'name', None): getattr(f, '@id', None) for f in getattr(first_record_set, 'field', [])}

# Try to pick likely field IDs. We'll assume typical clinical columns:
age_field_name = [k for k in fields if 'age' in k.lower()]
if not age_field_name:
    # Fallback: just pick first Float/Integer field
    numeric_field = next((getattr(f, '@id', None) for f in getattr(first_record_set, 'field', []) if getattr(f, 'dataType', '') in ('Float', 'Integer')), None)
else:
    numeric_field = fields[age_field_name[0]]

sex_field_name = [k for k in fields if 'sex' in k.lower() or 'gender' in k.lower()]
group_field = fields[sex_field_name[0]] if sex_field_name else None

print(f"Numeric field @id: {numeric_field}\nGroup (categorical) field @id: {group_field}")

df = dataframes[main_record_set_id]
# Defensive: skip if missing
if numeric_field not in df.columns:
    print(f"Field {numeric_field} not in columns; can't proceed with EDA. Columns: {df.columns.tolist()}")
else:
    # Cast to numeric (errors="coerce" will make missing strings into NaN)
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors="coerce")
    threshold = 50
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} rows")
    display_cols = [numeric_field]
    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    display_cols.append(f"{numeric_field}_normalized")
    print(filtered_df[display_cols].head())
    # Grouped statistics by sex/gender if present
    if group_field and group_field in df.columns:
        group_stats = filtered_df.groupby(group_field)[numeric_field].describe()
        print(f"\nGrouped statistics by {group_field}:\n", group_stats)

## 5. Visualization

Visualize a numeric distribution (e.g., Age) and relationships by a key grouping variable (e.g., Sex, if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Use previous variable names
if numeric_field in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=15, color='steelblue')
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel("Count")
    plt.show()
    
    if group_field and group_field in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field, palette='Set2')
        plt.title(f"{numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion

- We loaded and explored the FAIR² dataset using `mlcroissant`, referencing metadata entities by their `@id` according to the Croissant schema standard.
- We identified record sets and their fields, then loaded the clinical dataset into a DataFrame.
- We performed basic data cleaning, normalization, and grouped statistics using the field `@id` references only.
- Visualizations of numeric variables and group comparisons provided insights into the dataset's clinical attributes.

**For production or publication-level analysis, always carefully review the Croissant schema's field documentation to interpret each variable's meaning reliably.**